# OMol25 — browse, search and visualise

The OMol25 split in `data/OMol25/train_4M` is 3,986,754 molecular structures
in 80 LMDB shards, ~20 GB. This notebook is the way in: pull any structure out
of it **by index, by formula, by SMILES, by provenance or by any combination of
its metadata**, and look at it in 3D with py3Dmol.

Everything runs off two objects from `goal.ml.data.datasets.aselmdb`:

| | |
|---|---|
| `ASELMDBReader` | random access to the shards — a row dict or an `ase.Atoms` for any structure |
| `ASELMDBCatalogue` | a compact per-row summary (elements, atom count, energy, charge, spin, formula, sub-corpus) that makes searching 4M rows a numpy operation |

**Before the first run**, build the catalogues (one ~8 s scan per shard,
cached beside it — do them in parallel once and never again):

```bash
python scripts/omol25_prepare.py --root data/OMol25/train_4M --workers 8
```

Without that, the first cell below scans all 80 shards itself and takes about
ten minutes.

## Open the split

In [ ]:
from __future__ import annotations

import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import py3Dmol
import rootutils

rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from goal.ml.data.datasets.aselmdb import (
    ASELMDBCatalogue,
    ASELMDBReader,
    parse_formula,
    row_to_atoms,
)
from ase.data import chemical_symbols

SPLIT = Path("../data/OMol25/train_4M")
if not SPLIT.is_dir():                      # running from the repo root
    SPLIT = Path("data/OMol25/train_4M")

reader = ASELMDBReader(SPLIT)
catalogue = reader.catalogue(progress=print)   # cached; prints only on a cold scan

print(f"{len(reader):>12,} structures in {len(reader.files)} shards")
print(f"{len(catalogue):>12,} rows in the catalogue")

## What is in it

The catalogue answers this without decoding a single geometry — the summary
was built once and cached, so these are numpy reductions over arrays.

In [ ]:
def overview(cat: ASELMDBCatalogue, title: str = "whole split") -> None:
    """Print the shape of a catalogue: corpora, sizes, charges, chemistry."""
    print(f"{title}: {len(cat):,} structures\n")

    corpora = Counter(cat.data_id.tolist())
    width = max(len(k) for k in corpora)
    for name, count in corpora.most_common():
        print(f"  {name:<{width}}  {count:>9,}  {count / len(cat):6.1%}")

    sizes = cat.num_atoms
    print(f"\n  atoms/structure : min {sizes.min()}  median {int(np.median(sizes))}"
          f"  p95 {int(np.percentile(sizes, 95))}  max {sizes.max()}")
    print(f"  distinct formulas: {len(np.unique(cat.formula_codes)):,}")

    charges = Counter(cat.charge.tolist())
    spins = Counter(cat.spin.tolist())
    print(f"  charge          : {min(charges)} … {max(charges)}"
          f"   (neutral = {charges[0] / len(cat):.1%})")
    print(f"  spin            : {min(spins)} … {max(spins)}"
          f"   (singlet = {spins[1] / len(cat):.1%})")

    energy = cat.energy[np.isfinite(cat.energy)]
    print(f"  energy / eV     : {energy.min():,.0f} … {energy.max():,.0f}"
          f"   (raw wB97M-V total energies)")

    present = cat.unique_elements()
    print(f"\n  {len(present)} elements: "
          + " ".join(chemical_symbols[z] for z in present))


overview(catalogue)

Two things in that output shape every training decision on this corpus:

* **83 elements, charges −8…+9, spins 1…11.** A model whose input is
  `(Z, positions)` cannot distinguish a cation from its neutral parent at the
  same geometry, and a per-element-pair artisan bank over 83 elements is 3,486
  experts per round.
* **Energies of order 10⁴ eV.** These are raw total energies, so a per-element
  baseline is not a nicety — see `atomic_energies` in
  `configs/ml/nf_simurgh_omol25.yaml`.

That is why the training config filters. Here is the subset it selects, read
straight out of the config so this notebook cannot drift from it:

In [ ]:
from omegaconf import OmegaConf

from goal import CONFIGS_DIR

CONFIG = OmegaConf.load(Path(CONFIGS_DIR) / "ml" / "nf_simurgh_omol25.yaml")
FILTERS = CONFIG.data


def training_subset(cat: ASELMDBCatalogue) -> ASELMDBCatalogue:
    """The rows `configs/ml/nf_simurgh_omol25.yaml` would train on.

    Mirrors ``ASELMDBDataset``'s own filter, minus ``max_samples`` — this is
    the pool the per-shard cap is drawn from.
    """
    keep = cat.element_mask(
        allowed=FILTERS.allowed_elements,
        excluded=FILTERS.excluded_elements,
    )
    keep &= np.isfinite(cat.energy)
    if FILTERS.charges is not None:
        keep &= np.isin(cat.charge, np.asarray(FILTERS.charges, dtype=np.int16))
    if FILTERS.spins is not None:
        keep &= np.isin(cat.spin, np.asarray(FILTERS.spins, dtype=np.int16))
    if FILTERS.data_ids is not None:
        keep &= np.isin(cat.data_id, np.asarray(FILTERS.data_ids, dtype=str))
    if FILTERS.max_atoms is not None:
        keep &= cat.num_atoms <= FILTERS.max_atoms
    if FILTERS.min_atoms is not None:
        keep &= cat.num_atoms >= FILTERS.min_atoms
    return cat.select(keep)


subset = training_subset(catalogue)
overview(subset, "training subset")

## Pulling a structure out

A structure has three usable addresses:

* a **catalogue position** — the row's index in whichever catalogue you are
  holding, which is what every search below returns;
* a **reader position** — its index in the flat ordering over all shards, which
  is what `reader.read()` / `reader.atoms()` take;
* a **(shard, row id) address** — where it physically lives, stable forever.

`reader.positions_of(cat)` converts a catalogue to reader positions and
`reader.position_of(shard, row_id)` converts an address, so all three are
interchangeable. The helpers below hide that: they all return reader positions.

In [ ]:
def hill_formula(numbers) -> str:
    """Hill formula for a list of atomic numbers — the catalogue's convention."""
    from ase.symbols import Symbols

    return Symbols(np.asarray(numbers, dtype=int)).get_chemical_formula(mode="hill")


def describe(position: int) -> str:
    """One-line caption for a reader position."""
    row = reader.read(position)
    extras = row.get("data") or {}
    shard, row_id = reader.address(position)
    return (
        f"{hill_formula(row['numbers'])}"
        f"  {len(row['numbers'])} atoms"
        f"  q={extras.get('charge', '?')} S={extras.get('spin', '?')}"
        f"  E={row['energy']:,.3f} eV"
        f"  [{extras.get('data_id', '?')}]"
        f"  ({shard.name}#{row_id})"
    )


def by_address(shard: str, row_id: int) -> int:
    """Reader position of a (shard file name, row id) address."""
    match = [f for f in reader.files if f.name == shard or str(f) == shard]
    if not match:
        raise KeyError(f"{shard!r} is not a shard of this split")
    return reader.position_of(match[0], row_id)


def where(
    cat: ASELMDBCatalogue | None = None,
    formula: str | None = None,
    contains: list[int] | None = None,
    only: list[int] | None = None,
    data_id: str | None = None,
    charge: int | None = None,
    spin: int | None = None,
    atoms: tuple[int, int] | None = None,
    limit: int | None = 20,
) -> np.ndarray:
    """Reader positions of every row matching the given metadata.

    Every argument is optional and they compose; all of it runs on the
    catalogue's arrays, so searching the full 4M rows costs milliseconds.
    """
    cat = catalogue if cat is None else cat
    keep = np.ones(len(cat), dtype=bool)
    if formula is not None:
        target = hill_formula(
            [z for z, n in parse_formula(formula).items() for _ in range(n)]
        )
        keep &= cat.formula == target
    if only is not None:
        keep &= cat.element_mask(allowed=only)
    if contains is not None:
        needed = cat.element_mask(excluded=contains)   # rows WITHOUT any of them
        keep &= ~needed
    if data_id is not None:
        keep &= cat.data_id == data_id
    if charge is not None:
        keep &= cat.charge == charge
    if spin is not None:
        keep &= cat.spin == spin
    if atoms is not None:
        keep &= (cat.num_atoms >= atoms[0]) & (cat.num_atoms <= atoms[1])
    hits = cat.select(keep)
    if limit is not None and len(hits) > limit:
        hits = hits.select(np.arange(limit))
    return reader.positions_of(hits)


def sample(cat: ASELMDBCatalogue | None = None, n: int = 5, seed: int = 0) -> np.ndarray:
    """*n* random reader positions from a catalogue."""
    cat = catalogue if cat is None else cat
    picks = np.random.default_rng(seed).choice(len(cat), size=min(n, len(cat)), replace=False)
    return reader.positions_of(cat.select(np.sort(picks)))

A few examples — index, address, formula, metadata, random:

In [ ]:
print("by reader position 0        :", describe(0))
print("by address                  :", describe(by_address("data0000.aselmdb", 1)))
print()

print("caffeine (C8H10N4O2) in the split:")
for position in where(formula="C8H10N4O2", limit=3):
    print("   ", describe(position))
print()

print("small sulphur-containing biomolecules:")
for position in where(subset, contains=[16], data_id="biomolecules", atoms=(10, 25), limit=3):
    print("   ", describe(position))
print()

print("random draw from the training subset:")
for position in sample(subset, n=3, seed=7):
    print("   ", describe(position))

## Searching by SMILES

OMol25 stores coordinates, not connectivity — no SMILES field exists. So a
SMILES lookup is a two-stage search:

1. narrow to the rows whose **formula** matches the query (a numpy compare over
   the catalogue: exact, and it eliminates ~99.99% of the corpus);
2. **perceive** bonds for those candidates with RDKit's xyz2mol
   (`rdDetermineBonds`) and compare canonical SMILES.

Stage 2 is the expensive and fallible half — perception can fail on radicals,
hypervalent species and metal centres, and it is driven purely by geometry plus
total charge. Treat a match as "this geometry perceives as that molecule",
which is the strongest statement the data supports.

In [ ]:
def smiles_of(position: int, keep_hydrogens: bool = False) -> str | None:
    """Canonical SMILES perceived from a structure's coordinates.

    ``None`` when RDKit cannot assign a consistent bond order — expected for
    a few percent of the corpus, not a bug.
    """
    from rdkit import Chem
    from rdkit.Chem import rdDetermineBonds
    from rdkit.rdBase import BlockLogs

    row = reader.read(position)
    atoms = row_to_atoms(row)
    charge = int((row.get("data") or {}).get("charge", 0))
    block = f"{len(atoms)}\n\n" + "\n".join(
        f"{s} {x:.8f} {y:.8f} {z:.8f}"
        for s, (x, y, z) in zip(atoms.get_chemical_symbols(), atoms.positions)
    )
    with BlockLogs():                    # perception is chatty on stderr
        mol = Chem.MolFromXYZBlock(block)
        if mol is None:
            return None
        try:
            rdDetermineBonds.DetermineBonds(mol, charge=charge)
        except ValueError:
            return None
    if not keep_hydrogens:
        mol = Chem.RemoveHs(mol)
    return Chem.MolToSmiles(mol)


def by_smiles(
    query: str,
    cat: ASELMDBCatalogue | None = None,
    max_candidates: int = 400,
    limit: int = 10,
) -> list[int]:
    """Reader positions whose perceived structure matches *query*.

    ``max_candidates`` bounds stage 2: perception is ~0.5 ms for a 30-atom
    molecule but can take seconds for a large flexible one, so the number of
    structures it is allowed to try is capped rather than left to the size of
    the formula bucket.
    """
    from rdkit import Chem

    cat = catalogue if cat is None else cat
    query_mol = Chem.MolFromSmiles(query)
    if query_mol is None:
        raise ValueError(f"RDKit could not parse the query SMILES {query!r}")
    wanted = Chem.MolToSmiles(query_mol)
    numbers = [a.GetAtomicNum() for a in Chem.AddHs(query_mol).GetAtoms()]

    candidates = reader.positions_of(cat.select(cat.formula == hill_formula(numbers)))
    print(f"{len(candidates):,} structures share the formula {hill_formula(numbers)}"
          f"; perceiving up to {min(len(candidates), max_candidates):,}")

    found: list[int] = []
    for position in candidates[:max_candidates]:
        if smiles_of(position) == wanted:
            found.append(int(position))
            if len(found) >= limit:
                break
    return found


hits = by_smiles("CC(=O)Oc1ccccc1C(=O)O", cat=subset)     # aspirin
print(f"\n{len(hits)} confirmed match(es)")
for position in hits[:3]:
    print("   ", describe(position))

Note how few formula-matched candidates survive stage 2: OMol25 is full of isomers, so sharing a formula says very little. A search that returns nothing usually means the molecule is not in the corpus — 10 structures share caffeine's C8H10N4O2 and none of them perceive as caffeine.

The other direction — what does an arbitrary row perceive as?

In [ ]:
for position in sample(subset, n=6, seed=3):
    print(f"{describe(position)}\n      SMILES: {smiles_of(position)}")

## Looking at it — py3Dmol

`show()` renders one structure. Three things it can overlay:

* **`forces=True`** — the DFT forces as arrows. These are labels the model is
  trained against, so seeing them is seeing the target.
* **`color_by="fragment"`** — atoms coloured by the fragment decomposition
  `data.fragment_scheme` produces. This is what NF_SIMURGH actually pools onto
  its virtual nodes; if the colouring looks chemically arbitrary, the
  virtual-node channel is being fed something arbitrary.
* **`label=True`** — the formula drawn in the view (useful in grids).

In [ ]:
# A colourblind-safe qualitative palette, reused for fragments.
FRAGMENT_COLORS = [
    "#4477aa", "#ee6677", "#228833", "#ccbb44", "#66ccee",
    "#aa3377", "#bbbbbb", "#ee8866", "#44bb99", "#997700",
]


def xyz_block(atoms) -> str:
    """An XYZ block for py3Dmol's `addModel`."""
    lines = [str(len(atoms)), ""]
    lines += [
        f"{s} {x:.6f} {y:.6f} {z:.6f}"
        for s, (x, y, z) in zip(atoms.get_chemical_symbols(), atoms.positions)
    ]
    return "\n".join(lines)


def fragment_labels(position: int, scheme: str | None = None) -> np.ndarray:
    """Fragment ids for a structure, using the training config's settings."""
    import torch

    from goal.ml.data.fragments import compute_fragment_index

    row = reader.read(position)
    labels = compute_fragment_index(
        positions=torch.tensor(np.asarray(row["positions"]), dtype=torch.float64),
        atomic_numbers=torch.tensor(np.asarray(row["numbers"]), dtype=torch.long),
        covalent_cutoff=float(FILTERS.fragment_covalent_cutoff),
        cell=None,
        pbc=None,
        scheme=scheme or str(FILTERS.fragment_scheme),
        charge=int((row.get("data") or {}).get("charge", 0)),
        on_failure="fallback",
    )
    return labels.numpy()


def show(
    position: int,
    view=None,
    viewer: tuple[int, int] | None = None,
    width: int = 560,
    height: int = 440,
    forces: bool = False,
    force_scale: float = 0.6,
    color_by: str = "element",
    scheme: str | None = None,
    label: bool = False,
    caption: bool = True,
):
    """Render one structure. Returns the py3Dmol view (call `.show()` on it)."""
    row = reader.read(position)
    atoms = row_to_atoms(row)
    target = {} if viewer is None else {"viewer": viewer}
    view = view or py3Dmol.view(width=width, height=height)

    view.addModel(xyz_block(atoms), "xyz", **target)
    view.setStyle({"stick": {"radius": 0.10}, "sphere": {"scale": 0.20}}, **target)

    if color_by == "fragment":
        labels = fragment_labels(position, scheme)
        for fragment in range(int(labels.max()) + 1):
            color = FRAGMENT_COLORS[fragment % len(FRAGMENT_COLORS)]
            view.setStyle(
                {"index": np.flatnonzero(labels == fragment).tolist()},
                {"stick": {"radius": 0.10, "color": color},
                 "sphere": {"scale": 0.22, "color": color}},
                **target,
            )
    elif color_by != "element":
        raise ValueError('color_by must be "element" or "fragment"')

    if forces:
        vectors = np.asarray(row["forces"], dtype=float)
        for (x, y, z), (fx, fy, fz) in zip(atoms.positions, vectors * force_scale):
            view.addArrow(
                {"start": {"x": float(x), "y": float(y), "z": float(z)},
                 "end": {"x": float(x + fx), "y": float(y + fy), "z": float(z + fz)},
                 "radius": 0.045, "radiusRatio": 2.5, "mid": 0.75, "color": "#cc3311"},
                **target,
            )

    if label:
        view.addLabel(
            hill_formula(row["numbers"]),
            {"backgroundColor": "white", "backgroundOpacity": 0.65,
             "fontColor": "black", "fontSize": 12,
             "position": {"x": float(atoms.positions[:, 0].mean()),
                          "y": float(atoms.positions[:, 1].max() + 1.5),
                          "z": float(atoms.positions[:, 2].mean())}},
            **target,
        )

    view.setBackgroundColor("white")
    view.zoomTo(**target)
    if caption:
        print(describe(position))
    return view


def show_many(
    positions,
    columns: int = 3,
    cell_width: int = 300,
    cell_height: int = 240,
    **kwargs,
):
    """A grid of structures, one per cell."""
    positions = [int(p) for p in positions]
    rows = (len(positions) + columns - 1) // columns
    view = py3Dmol.view(
        viewergrid=(rows, columns),
        width=cell_width * columns,
        height=cell_height * rows,
        linked=False,
    )
    for slot, position in enumerate(positions):
        show(position, view=view, viewer=(slot // columns, slot % columns),
             caption=False, **kwargs)
    return view

One molecule, plain — aspirin, which the SMILES search above located:

In [ ]:
aspirin = hits[0] if hits else int(where(subset, formula="C9H8O4", limit=1)[0])
show(aspirin).show()

The same molecule with its DFT forces — the labels the model fits:

In [ ]:
show(aspirin, forces=True, force_scale=1.5).show()

### Fragments — what NF_SIMURGH actually pools

The model sums per-atom energies onto one virtual node per fragment and lets
those nodes interact pairwise out to 15 Å. The fragment decomposition *is* the
model's coarse-grained picture of a structure, so it is worth looking at rather
than trusting.

The config uses `fragment_scheme: connected` — connected components of the
covalent distance graph. On a single molecule like aspirin that is one
fragment, and the virtual-node channel contributes nothing to it:

In [ ]:
for name in ("connected", "rdkit_components", "rotatable", "brics", "recap"):
    labels = fragment_labels(aspirin, scheme=name)
    print(f"  {name:<18} K={labels.max() + 1:>2}  sizes {np.bincount(labels).tolist()}")

`rotatable` is what cuts a single molecule into chemical groups:

In [ ]:
show(aspirin, color_by="fragment", scheme="rotatable").show()

But `connected` is not idle on OMol25, and that is why the config chooses it:
half this corpus is genuinely multi-species — electrolyte clusters, reacting
complexes, solvated ions — and there its fragments are whole molecules and
ions, with the virtual-node channel modelling the intermolecular interaction
between them across a window atom-level message passing cannot reach.

In [ ]:
def first_multi_species(
    cat: ASELMDBCatalogue,
    min_fragments: int = 3,
    corpus: str = "elytes",
    atoms: tuple[int, int] = (25, 60),
    tries: int = 200,
):
    """The first structure the config's scheme splits into >= N fragments."""
    band = np.flatnonzero(
        (cat.num_atoms >= atoms[0]) & (cat.num_atoms <= atoms[1]) & (cat.data_id == corpus)
    )
    for position in reader.positions_of(cat.select(band[:tries])):
        labels = fragment_labels(int(position))
        if labels.max() + 1 >= min_fragments:
            return int(position), labels
    raise LookupError("no multi-fragment structure in that band")


cluster, labels = first_multi_species(subset)
print(f"{describe(cluster)}\n      K={labels.max() + 1} fragments, "
      f"sizes {np.bincount(labels).tolist()}")
show(cluster, color_by="fragment").show()

A grid of random structures from the training subset, fragment-coloured under
the config's scheme. Single-colour cells are single molecules; the
multi-coloured ones are where the virtual-node channel does its work.

In [ ]:
show_many(sample(subset, n=6, seed=11), columns=3, color_by="fragment", label=True).show()

## Searching by provenance

Every row records where it came from — `data["source"]`, e.g.
`orbnet_denali/orbnet_CHEMBL1935278_conformers_.../orca.tar.zst`. That is how
you find every conformer of one upstream molecule, or everything drawn from a
particular source dataset.

Those strings are long, so the cached catalogue leaves them out by default.
Two ways to get at them:

* **`source_of(position)`** — read one row's source. Instant, no index needed.
* **`find_source(text, shards=N)`** — build a *source-carrying* catalogue for
  the first `N` shards (an ~8 s scan each, cached separately as
  `<shard>.goal-catalogue-src.npz`) and substring-search it. Start with a few
  shards; indexing all 80 for source search costs the same ten minutes as the
  original scan.

In [ ]:
def source_of(position: int) -> str:
    """The upstream provenance string for one structure."""
    return str((reader.read(position).get("data") or {}).get("source", ""))


def find_source(text: str, shards: int = 2, limit: int = 10) -> np.ndarray:
    """Reader positions whose source string contains *text*."""
    partial = ASELMDBReader(reader.files[:shards])
    with_source = partial.catalogue(include_source=True, progress=print)
    keep = np.char.find(with_source.source.astype(str), text) >= 0
    hits = with_source.select(keep)
    print(f"{len(hits):,} match(es) for {text!r} in the first {shards} shard(s)")
    if limit is not None and len(hits) > limit:
        hits = hits.select(np.arange(limit))
    return reader.positions_of(hits)


print("source of row 0:", source_of(0))

In [ ]:
# Every conformer of one ChEMBL molecule that OrbNet Denali contributed.
conformers = find_source("CHEMBL1935278", shards=1, limit=6)
for position in conformers:
    print("   ", describe(position))
if len(conformers):
    show_many(conformers, columns=3, label=True).show()

## How this reaches the model

`ASELMDBDataset` is the same reader plus the filters, emitting `AtomicGraph`
objects. Constructing it straight from the config section is the honest check
that what you have been browsing is what will train:

In [ ]:
import torch

from goal.ml.data.datasets.aselmdb import ASELMDBDataset

warnings.filterwarnings("ignore")           # rdkit fallbacks warn once each

dataset = ASELMDBDataset(
    root=SPLIT / reader.files[0].name,
    cutoff=float(CONFIG.data.cutoff),
    allowed_elements=list(FILTERS.allowed_elements),
    charges=list(FILTERS.charges),
    spins=list(FILTERS.spins),
    max_atoms=int(FILTERS.max_atoms),
    max_samples=64,
    sample_seed=int(FILTERS.sample_seed),
    compute_fragment_index=bool(FILTERS.compute_fragment_index),
    fragment_scheme=str(FILTERS.fragment_scheme),
    fragment_covalent_cutoff=float(FILTERS.fragment_covalent_cutoff),
    fragment_on_failure=str(FILTERS.fragment_on_failure),
    dtype=torch.float32,
)

graph = dataset[0]
print(graph)
print(f"\n  {graph.z.shape[0]} atoms, {graph.edge_index.shape[1]} edges at "
      f"{CONFIG.data.cutoff} A, {int(graph.fragment_index.max()) + 1} fragments")
print(f"  energy {graph.energy.item():,.3f} eV, max |force| "
      f"{graph.forces.abs().max():.3f} eV/A")

# dataset.catalogue is aligned with dataset[i], and dataset.atoms(i) gives the
# ase.Atoms behind it — so anything above can be pointed at a training row.
print(f"\n  dataset[0] is {dataset.catalogue.formula[0]} "
      f"from {dataset.catalogue.data_id[0]}")

In [ ]:
# ... and one full batch, the way Lightning will see it.
from torch_geometric.loader import DataLoader

batch = next(iter(DataLoader(dataset, batch_size=8, shuffle=False)))
print(batch)
print(f"\n  {int(batch.batch.max()) + 1} structures, {batch.z.shape[0]} atoms, "
      f"{batch.edge_index.shape[1]} edges, "
      f"{int(batch.fragment_index.max()) + 1} fragments total")

## Where to go next

* `configs/ml/nf_simurgh_omol25.yaml` — the training config, with the reasoning
  for every filter above written next to it.
* `scripts/omol25_prepare.py` — rebuilds the catalogues and re-derives
  `atomic_energies` / `scale` / `avg_num_neighbors` for a *different* filter
  set. Run it whenever you change `allowed_elements`, `charges`, `spins` or
  `max_atoms`; those four decide the energy baseline.
* `goal.ml.data.datasets.aselmdb` — `linear_reference_energies` and
  `residual_energy_scale` compute the baseline and the energy scale from a
  catalogue alone, without decoding geometry, if you want them for a subset you
  defined here.